# Evaluation and Fine-Tuning of EfficientLoFTR for Thermal-Optical Image Matching

**Task:** Evaluate the EfficientLoFTR (ELoFTR) image matching model on multimodal
(thermal-optical) image pairs, fine-tune it on a thermal-optical dataset, and
compare it against a classical feature matching pipeline (ORB + RANSAC).

**Dataset:** RoadScene (paired infrared / visible images), used as a smaller and
more practical substitute for LasHeR due to compute constraints.

**Reference:** [EfficientLoFTR project page](https://zju3dv.github.io/efficientloftr/) |
[GitHub repository](https://github.com/zju3dv/efficientloftr)

**Outputs covered in this notebook:**
- Feature correspondence visualizations
- Inlier ratio after RANSAC
- Pose / reprojection error (via Mean Matching Accuracy, MMA)
- Runtime and computational efficiency
- Strengths and weaknesses of each approach

---

The cell below contains **all** the imports needed for the rest of the notebook.
No other cell imports a library; this keeps the dependency list in one place and
makes the notebook easier to review and reproduce.


In [ ]:
# ============================================================
# LIBRARY / IMPORT CELL
# ------------------------------------------------------------
# Every import used anywhere in this notebook is centralized
# here. No other cell below contains an import statement.
# ============================================================

# --- Standard library ---
import os
import sys
import time
import gc
import random
from copy import deepcopy

# --- Numerical / computer vision / plotting ---
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# --- Data augmentation ---
import albumentations as A

# --- PyTorch ---
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim import Adam
from torch.amp import autocast, GradScaler
from tqdm.notebook import tqdm

# --- Make the EfficientLoFTR repository importable ---
# This must run BEFORE the "src.*" / "configs.*" imports below.
sys.path.append('./efficientloftr')

# --- EfficientLoFTR repository imports ---
# The repository exposes two different objects both called "LoFTR":
#   1) src.loftr.LoFTR        -> high-level wrapper, used for quick
#                                 pretrained zero-shot inference together
#                                 with full_default_cfg and reparameter().
#   2) src.loftr.loftr.LoFTR  -> the raw model class, used whenever we
#                                 build the model manually from a custom
#                                 config (training, fine-tuning, evaluation).
# To avoid a name collision, the first one is imported under the alias
# "LoFTR_Inference". The plain name "LoFTR" always refers to the raw
# model class (case 2), which is what most of this notebook uses.
from src.loftr import LoFTR as LoFTR_Inference, full_default_cfg, reparameter
from src.loftr.loftr import LoFTR
from src.utils.plotting import make_matching_figure
from src.config.default import get_cfg_defaults
from configs.loftr.eloftr_full import cfg as user_cfg
from src.losses.loftr_loss import LoFTRLoss

print("All libraries imported successfully.")


## 1. Dataset Structure Check

Before running any model, we inspect the folder structure of the RoadScene
dataset to confirm that the expected sub-folders (infrared images, visible
images, and ground-truth homographies) are present and correctly organized.


In [ ]:
def print_directory_tree(startpath, max_files_per_dir=3):
    """
    Prints the directory structure of the given path.
    Limits the number of printed files per directory to keep output short.
    """
    print(f"Structure of: {startpath}\n" + "-"*30)
    
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}📂 {os.path.basename(root)}/")
        
        subindent = ' ' * 4 * (level + 1)
        
        # Sort files so we see them in a logical order
        files.sort()
        
        for i, f in enumerate(files):
            if i < max_files_per_dir:
                print(f"{subindent}📄 {f}")
            elif i == max_files_per_dir:
                print(f"{subindent}📄 ... and {len(files) - max_files_per_dir} more files")
                break

# Replace this string with the actual path to your RoadScene folder
dataset_folder_path = "./RoadScene" 

print_directory_tree(dataset_folder_path)

## 2. Classical Baseline: ORB + RANSAC

This cell implements the classical feature matching pipeline required by the
task: **ORB** keypoint detection and description, **Brute-Force Hamming
matching**, and **RANSAC**-based geometric verification using a homography
model.

We report the total number of matches, the number of RANSAC inliers, the
inlier ratio, the runtime, and we save a visualization of the inlier matches
for later use in the presentation.


In [ ]:
# Step 5: Data Loading
img_name = "FLIR_00006.jpg"
path_thermal = os.path.join("./RoadScene/cropinfrared", img_name)
path_optical = os.path.join("./RoadScene/crop_LR_visible", img_name)

img_thermal = cv2.imread(path_thermal, cv2.IMREAD_GRAYSCALE)
img_optical = cv2.imread(path_optical, cv2.IMREAD_GRAYSCALE)

start_time = time.time()

# Step 6: Feature Extraction & Matching (ORB)
orb = cv2.ORB_create(nfeatures=2000)
kp1, des1 = orb.detectAndCompute(img_thermal, None)
kp2, des2 = orb.detectAndCompute(img_optical, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des1, des2)
matches = sorted(matches, key=lambda x: x.distance)

# Step 7: RANSAC Verification
src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
matchesMask = mask.ravel().tolist()

end_time = time.time()

# Step 8: Baseline Metrics
inlier_ratio = sum(matchesMask) / len(matchesMask) if len(matchesMask) > 0 else 0

print(f"Classical ORB Baseline Results:")
print(f"Total Matches: {len(matches)}")
print(f"Inliers after RANSAC: {sum(matchesMask)}")
print(f"Inlier Ratio: {inlier_ratio:.2%}")
print(f"Runtime: {end_time - start_time:.4f} seconds")

# Save Visualization for Presentation
draw_params = dict(matchColor=(0, 255, 0), singlePointColor=None, matchesMask=matchesMask, flags=2)
img_matches = cv2.drawMatches(img_thermal, kp1, img_optical, kp2, matches, None, **draw_params)

plt.figure(figsize=(15, 5))
plt.imshow(img_matches, cmap='gray')
plt.title(f"ORB + RANSAC Inliers | {img_name}")
plt.axis('off')

# This saves the image to your current folder
save_path = "orb_baseline_00006.png"
plt.savefig(save_path, bbox_inches='tight')
print(f"\nVisualization saved to {save_path} for your presentation!")
plt.show()

## 3. Zero-Shot EfficientLoFTR (Pretrained, No Fine-Tuning)

Here we run the **pretrained** EfficientLoFTR model (trained on MegaDepth,
an outdoor optical dataset) directly on a thermal-optical image pair, without
any fine-tuning. This tells us how well the model generalizes to a new,
unseen modality (thermal images) out of the box.

The same RANSAC-based verification used for ORB is applied here so that the
inlier ratio is directly comparable between the two methods.


In [ ]:
# 1. Initialize the matcher
_default_cfg = deepcopy(full_default_cfg)
matcher = LoFTR_Inference(config=_default_cfg)

# Load the weights you just downloaded
weight_path = "./efficientloftr/weights/eloftr_outdoor.ckpt"
matcher.load_state_dict(torch.load(weight_path, weights_only=False)['state_dict'])
matcher = reparameter(matcher).eval().cuda()

# 2. Load and Preprocess the same images
img_name = "FLIR_00006.jpg"
path_thermal = os.path.join("./RoadScene/cropinfrared", img_name)
path_optical = os.path.join("./RoadScene/crop_LR_visible", img_name)

img_thermal_raw = cv2.imread(path_thermal, cv2.IMREAD_GRAYSCALE)
img_optical_raw = cv2.imread(path_optical, cv2.IMREAD_GRAYSCALE)

# ELoFTR requires dimensions divisible by 32
img0_raw = cv2.resize(img_thermal_raw, (img_thermal_raw.shape[1]//32*32, img_thermal_raw.shape[0]//32*32))
img1_raw = cv2.resize(img_optical_raw, (img_optical_raw.shape[1]//32*32, img_optical_raw.shape[0]//32*32))

img0 = torch.from_numpy(img0_raw)[None][None].cuda() / 255.
img1 = torch.from_numpy(img1_raw)[None][None].cuda() / 255.
batch = {'image0': img0, 'image1': img1}

# 3. Run Inference (Steps 9 & 10)
start_time = time.time()
with torch.no_grad():
    matcher(batch)
    mkpts0 = batch['mkpts0_f'].cpu().numpy()
    mkpts1 = batch['mkpts1_f'].cpu().numpy()
    mconf = batch['mconf'].cpu().numpy()
end_time = time.time()

# --- THE FIX: Filter the arrays using the RANSAC mask ---
if len(mkpts0) >= 4:
    H_eloftr, mask_eloftr = cv2.findHomography(
        mkpts0, mkpts1, cv2.RANSAC, 3.0
    )
    
    # Convert mask to boolean and filter the points
    valid_mask = mask_eloftr.ravel() == 1
    mkpts0_filtered = mkpts0[valid_mask]
    mkpts1_filtered = mkpts1[valid_mask]
    mconf_filtered = mconf[valid_mask]
    
    inliers = valid_mask.sum()
    inlier_ratio_eloftr = inliers / len(mkpts0)
    print(f"Inliers after RANSAC: {inliers}")
    print(f"Inlier Ratio: {inlier_ratio_eloftr:.2%}")
else:
    # Fallback if too few points are found to run RANSAC
    mkpts0_filtered, mkpts1_filtered, mconf_filtered = mkpts0, mkpts1, mconf
    
print(f"ELoFTR Zero-Shot Results:")
print(f"Total Matches Found: {len(mkpts0)}")
print(f"Runtime: {end_time - start_time:.4f} seconds")

# 4. Visualization (Now using the filtered points!)
color = cm.jet(mconf_filtered)
text = ['EfficientLoFTR Zero-Shot (RANSAC)', f'Inliers: {len(mkpts0_filtered)}']
fig = make_matching_figure(img0_raw, img1_raw, mkpts0_filtered, mkpts1_filtered, color, text=text)

# Save the output
save_path = "eloftr_zeroshot_00006.png"
fig.savefig(save_path, bbox_inches='tight')
print(f"\nVisualization saved to {save_path} for your presentation!")
plt.show()

## 4. Custom Dataset Class for Fine-Tuning

To fine-tune EfficientLoFTR on thermal-optical data, we need a `Dataset` that:
- Loads paired infrared and visible images.
- Applies pixel-level augmentation (brightness/contrast, Gaussian noise).
- Applies a random homography warp to the visible image, so that the exact
  ground-truth transformation (`H_gt`) between the two images is always known.
  This ground truth is what allows us to supervise the model during training
  and to compute the reprojection error during evaluation.

The dataset is then split into training, validation, and test subsets, and
wrapped in PyTorch `DataLoader`s.

**Note:** the `IR_DIR` and `VIS_DIR` paths below are local paths from the
original development machine. Update them to match your own environment
before running this cell.


In [ ]:
class RoadSceneDataset(Dataset):
    """
    Pairs crop_LR_visible (query) with cropinfrared (reference).
    Applies a random homography warp to the visible image to force the model
    to learn real cross-modal feature matching.
    """
    def __init__(self, ir_dir, vis_dir, image_size=(480, 640), augment=True):
        self.ir_dir   = ir_dir
        self.vis_dir  = vis_dir
        self.augment  = augment
        self.h, self.w = image_size

        # Only keep files present in BOTH folders
        ir_files  = set(os.listdir(ir_dir))
        vis_files = set(os.listdir(vis_dir))
        self.files = sorted(ir_files & vis_files)
        assert len(self.files) > 0, "No paired files found!"

        # Pixel-level augmentation
        self.pixel_aug_ir  = A.Compose([
            A.RandomBrightnessContrast(p=0.5),
            A.GaussNoise(p=0.3),
        ])
        self.pixel_aug_vis = A.Compose([
            A.RandomBrightnessContrast(p=0.5),
            A.GaussNoise(p=0.3),
        ])

    def __len__(self):
        return len(self.files)

    def _load_gray(self, path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        assert img is not None, f"Cannot read: {path}"
        img = cv2.resize(img, (self.w, self.h))
        return img

    def _generate_random_homography(self, h, w, max_shift=0.05):
        """Generates a random perspective warp matrix."""
        pts1 = np.float32([[0, 0], [w, 0], [0, h], [w, h]])
        # Randomly shift the 4 corners by up to 5% of the image size
        shifts = np.random.uniform(-max_shift, max_shift, pts1.shape).astype(np.float32)
        shifts[:, 0] *= w
        shifts[:, 1] *= h
        pts2 = pts1 + shifts
        H = cv2.getPerspectiveTransform(pts1, pts2)
        return H

    def __getitem__(self, idx):
        fname = self.files[idx]

        ir  = self._load_gray(os.path.join(self.ir_dir,  fname))
        vis = self._load_gray(os.path.join(self.vis_dir, fname))

        # Default ground truth is the identity matrix (no movement)
        H_gt = np.eye(3, dtype=np.float32)

        if self.augment:
            # 1. Pixel Augmentations
            ir  = self.pixel_aug_ir(image=ir)['image']
            vis = self.pixel_aug_vis(image=vis)['image']

            # 2. Spatial Augmentation (Warp)
            H_gt = self._generate_random_homography(self.h, self.w)
            vis = cv2.warpPerspective(vis, H_gt, (self.w, self.h), borderMode=cv2.BORDER_REFLECT)

        # To tensor [1, H, W] float32 in [0, 1]
        ir_t  = torch.from_numpy(ir).float().unsqueeze(0) / 255.
        vis_t = torch.from_numpy(vis).float().unsqueeze(0) / 255.

        return {
            'image0': ir_t,   # infrared
            'image1': vis_t,  # visible
            'H_gt': torch.from_numpy(H_gt).float(), # <-- The exact warp matrix!
            'dataset_name': 'RoadScene',
            'scene_id': fname,
            'pair_id': idx,
        }

# ── Paths ──────────────────────────────────────────────────────────────────────
IR_DIR  = r'D:\Courses\Deep learning\ELoFTR Hasan f.Ates Task\RoadScene\cropinfrared'
VIS_DIR = r'D:\Courses\Deep learning\ELoFTR Hasan f.Ates Task\RoadScene\crop_LR_visible'
IMAGE_SIZE = (480, 640)   

# ── Build full dataset ─────────────────────────────────────────────────────────
base_train = RoadSceneDataset(IR_DIR, VIS_DIR, image_size=IMAGE_SIZE, augment=True)
base_eval  = RoadSceneDataset(IR_DIR, VIS_DIR, image_size=IMAGE_SIZE, augment=False)

n       = len(base_train)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
n_test  = n - n_train - n_val

torch.manual_seed(42)
indices = torch.randperm(n).tolist()

train_ds = Subset(base_train, indices[:n_train])
val_ds   = Subset(base_eval,  indices[n_train:n_train+n_val])
test_ds  = Subset(base_eval,  indices[n_train+n_val:])

print(f"Total: {n}  |  Train: {n_train}  Val: {n_val}  Test: {n_test}")

# ── DataLoaders ────────────────────────────────────────────────────────────────
BATCH_SIZE  = 4     # Kept at 1 for strict memory safety
NUM_WORKERS = 0     
PIN_MEMORY  = True 

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          drop_last=True)

val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

batch = next(iter(train_loader))
print("image0 shape:", batch['image0'].shape)
print("H_gt shape:", batch['H_gt'].shape)

## 5. Fine-Tuning — Stage 1: Train the CNN Backbone Only

Fine-tuning is done in two stages to keep training stable, following common
practice when adapting a pretrained network to a new domain:

- **Stage 1 (this cell):** the Transformer / attention layers are **frozen**,
  and only the CNN backbone is trained. This lets the backbone adapt to the
  thermal-optical domain gap first, without destabilizing the already-trained
  matching layers.
- The ground-truth homography is converted into a coarse-level matching
  supervision matrix (`custom_spvs_homography`), since EfficientLoFTR is
  trained with a "which patch matches which patch" supervision signal.
- The best model (lowest validation loss) is saved as
  `eloftr_roadscene_stage1_best.pt`.


In [ ]:
# Clean slate
torch.cuda.empty_cache()
gc.collect()

# -------------------------------------------------------------------------
# 1. ARCHITECTURE & LOSS CONFIGURATION
# -------------------------------------------------------------------------
cfg = get_cfg_defaults()
cfg.merge_from_other_cfg(user_cfg)

cfg.LOFTR.LOSS.TYPE = 'homography'
cfg.LOFTR.MATCH_COARSE.TRAIN_COARSE_PERCENT = 0.0
cfg.LOFTR.MATCH_FINE.ENABLED = False
cfg.LOFTR.LOSS.FINE_WEIGHT = 0.0 
cfg.LOFTR.COARSE.NPE = [480, 640, 480, 640]

def lower_config(yacs_cfg):
    lower_dict = {}
    for k, v in yacs_cfg.items():
        if hasattr(v, 'items'):
            lower_dict[k.lower()] = lower_config(v)
        else:
            lower_dict[k.lower()] = v
    return lower_dict

global_cfg_lower = lower_config(cfg)

print("Initializing Model & Loss...")
model = LoFTR(config=global_cfg_lower['loftr']).cuda()

# --- THE MISSING FIX: LOAD THE PRE-TRAINED MEGA-DEPTH WEIGHTS FIRST ---
print("Loading pre-trained ELoFTR weights...")
weight_path = "./efficientloftr/weights/eloftr_outdoor.ckpt"
model.load_state_dict(torch.load(weight_path, weights_only=False)['state_dict'])
# ----------------------------------------------------------------------

loss_fn = LoFTRLoss(config=global_cfg_lower)

# -------------------------------------------------------------------------
# 2. STAGE 1 SETUP: FREEZE TRANSFORMER, TRAIN BACKBONE
# -------------------------------------------------------------------------
for name, param in model.named_parameters():
    if 'backbone' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
scaler = GradScaler('cuda')
print("Stage 1 Ready: Transformer frozen, CNN Backbone unfrozen.")

# -------------------------------------------------------------------------
# 3. CUSTOM SUPERVISION (Homography Projection)
# -------------------------------------------------------------------------
def custom_spvs_homography(batch):
    device = batch['image0'].device
    B, _, H, W = batch['image0'].shape
    Hc, Wc = H // 8, W // 8
    L = Hc * Wc
    
    ys, xs = torch.meshgrid(torch.arange(Hc, device=device), torch.arange(Wc, device=device), indexing='ij')
    grid_c = torch.stack([xs.flatten(), ys.flatten()], dim=-1).float()
    
    grid_img = grid_c * 8 + 4
    grid_img_h = torch.cat([grid_img, torch.ones_like(grid_img[:, :1])], dim=-1)
    
    conf_matrix_gt = torch.zeros(B, L, L, device=device)
    b_ids, i_ids, j_ids = [], [], []
    
    for b in range(B):
        H_mat = batch['H_gt'][b]
        
        warped_img_h = (H_mat @ grid_img_h.T).T
        z = warped_img_h[:, 2:]
        warped_img = warped_img_h[:, :2] / (z + 1e-8)
        
        warped_c = (warped_img - 4) / 8
        warped_c_idx = warped_c.round().long()
        
        valid_mask = (warped_c_idx[:, 0] >= 0) & (warped_c_idx[:, 0] < Wc) & \
                     (warped_c_idx[:, 1] >= 0) & (warped_c_idx[:, 1] < Hc)
        
        i_valid = torch.arange(L, device=device)[valid_mask]
        j_valid = warped_c_idx[valid_mask, 1] * Wc + warped_c_idx[valid_mask, 0]
        
        conf_matrix_gt[b, i_valid, j_valid] = 1.0
        
        b_ids.append(torch.full_like(i_valid, b))
        i_ids.append(i_valid)
        j_ids.append(j_valid)
        
    batch['conf_matrix_gt'] = conf_matrix_gt
    
    batch['spv_b_ids'] = torch.cat(b_ids) if len(b_ids) > 0 else torch.empty(0, device=device, dtype=torch.long)
    batch['spv_i_ids'] = torch.cat(i_ids) if len(i_ids) > 0 else torch.empty(0, device=device, dtype=torch.long)
    batch['spv_j_ids'] = torch.cat(j_ids) if len(j_ids) > 0 else torch.empty(0, device=device, dtype=torch.long)
    
    num_valid_matches = len(batch['spv_b_ids'])
    batch['conf_matrix_error_gt'] = torch.ones(num_valid_matches, device=device)
    batch['spv_w_pt0_i'] = torch.ones(num_valid_matches, device=device)

    # --- ALL REQUIRED DUMMY TENSORS ---
    batch['conf_matrix_f'] = torch.zeros(1, 1, 1, device=device)
    batch['conf_matrix_f_gt'] = torch.zeros(1, 1, 1, device=device)
    batch['conf_matrix_f_error_gt'] = torch.zeros(1, 1, 1, device=device)
    
    batch['expec_f'] = torch.zeros(1, 2, device=device)
    batch['expec_f_gt'] = torch.zeros(1, 2, device=device)

# -------------------------------------------------------------------------
# 4. THE STANDALONE TRAINING LOOP WITH VALIDATION
# -------------------------------------------------------------------------
EPOCHS = 25
best_val_loss = float('inf')

model.train()
model.coarse_matching.eval() 
model.fine_matching.eval()   

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    total_train_loss = 0.0
    pbar_train = tqdm(train_loader, desc=f"Train Epoch {epoch+1}/{EPOCHS}")
    
    for batch in pbar_train:
        for k in ['image0', 'image1', 'H_gt']:
            batch[k] = batch[k].cuda()
            
        optimizer.zero_grad()
        
        with autocast('cuda'): 
            model(batch) 
            custom_spvs_homography(batch)
            loss_fn(batch)
            loss = batch['loss'] 
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item()
        pbar_train.set_postfix({'Loss': f"{loss.item():.4f}"})
        del batch
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --- VALIDATION PHASE ---
    model.eval() 
    total_val_loss = 0.0
    
    with torch.no_grad(): 
        for val_batch in val_loader:
            for k in ['image0', 'image1', 'H_gt']:
                val_batch[k] = val_batch[k].cuda()
                
            with autocast('cuda'):
                model(val_batch)
                custom_spvs_homography(val_batch)
                loss_fn(val_batch)
                val_loss = val_batch['loss']
                
            total_val_loss += val_loss.item()
            del val_batch
            
    avg_val_loss = total_val_loss / len(val_loader)
    print(f"End of Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # --- CHECKPOINT SAVING ---
    if avg_val_loss < best_val_loss:
        print(f"--> Validation loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f}. Saving best model!")
        best_val_loss = avg_val_loss
        checkpoint_path = "eloftr_roadscene_stage1_best.pt"
        torch.save(model.state_dict(), checkpoint_path)
    else:
        print(f"--> Validation loss did not improve.")
    
    model.train()
    model.coarse_matching.eval() 
    model.fine_matching.eval()
    torch.cuda.empty_cache()
    
print(f"Stage 1 Complete! Best validation loss achieved: {best_val_loss:.4f}")

## 6. Fine-Tuning — Stage 2: End-to-End Training

In this second stage, we load the best Stage 1 checkpoint and **unfreeze the
entire network**, including the Transformer layers. This allows the full
model — backbone and attention layers together — to adapt jointly to the
thermal-optical matching task.

The training loop, loss, and supervision function are the same as in Stage 1.
The best model from this stage is saved as `eloftr_roadscene_stage2_best.pt`,
and this is the final fine-tuned model used for evaluation.


In [ ]:
# Clean slate
torch.cuda.empty_cache()
gc.collect()

# -------------------------------------------------------------------------
# 1. ARCHITECTURE & LOSS CONFIGURATION
# -------------------------------------------------------------------------
cfg = get_cfg_defaults()
cfg.merge_from_other_cfg(user_cfg)

cfg.LOFTR.LOSS.TYPE = 'homography'
cfg.LOFTR.MATCH_COARSE.TRAIN_COARSE_PERCENT = 0.0
cfg.LOFTR.MATCH_FINE.ENABLED = False
cfg.LOFTR.LOSS.FINE_WEIGHT = 0.0 
cfg.LOFTR.COARSE.NPE = [480, 640, 480, 640]

def lower_config(yacs_cfg):
    lower_dict = {}
    for k, v in yacs_cfg.items():
        if hasattr(v, 'items'):
            lower_dict[k.lower()] = lower_config(v)
        else:
            lower_dict[k.lower()] = v
    return lower_dict

global_cfg_lower = lower_config(cfg)

print("Initializing Model & Loss...")
model = LoFTR(config=global_cfg_lower['loftr']).cuda()
loss_fn = LoFTRLoss(config=global_cfg_lower)

# -------------------------------------------------------------------------
# 2. STAGE 2 SETUP: LOAD BEST WEIGHTS AND UNFREEZE ALL
# -------------------------------------------------------------------------
print("Loading Best Stage 1 weights...")
# Load the perfectly tuned CNN backbone from Stage 1
model.load_state_dict(torch.load('eloftr_roadscene_stage1_best.pt'))

# Unfreeze every single parameter in the network so the Transformer can learn
for param in model.parameters():
    param.requires_grad = True 

# Using the safe 1e-5 learning rate for end-to-end training
optimizer = Adam(model.parameters(), lr=1e-5)
scaler = GradScaler('cuda')
print("Stage 2 Ready: Full network unfrozen. Transformer is active.")

# -------------------------------------------------------------------------
# 3. SUPERVISION (Our perfected homography function)
# -------------------------------------------------------------------------
def custom_spvs_homography(batch):
    device = batch['image0'].device
    B, _, H, W = batch['image0'].shape
    Hc, Wc = H // 8, W // 8
    L = Hc * Wc
    
    ys, xs = torch.meshgrid(torch.arange(Hc, device=device), torch.arange(Wc, device=device), indexing='ij')
    grid_c = torch.stack([xs.flatten(), ys.flatten()], dim=-1).float()
    
    grid_img = grid_c * 8 + 4
    grid_img_h = torch.cat([grid_img, torch.ones_like(grid_img[:, :1])], dim=-1)
    
    conf_matrix_gt = torch.zeros(B, L, L, device=device)
    b_ids, i_ids, j_ids = [], [], []
    
    for b in range(B):
        H_mat = batch['H_gt'][b]
        
        warped_img_h = (H_mat @ grid_img_h.T).T
        z = warped_img_h[:, 2:]
        warped_img = warped_img_h[:, :2] / (z + 1e-8)
        
        warped_c = (warped_img - 4) / 8
        warped_c_idx = warped_c.round().long()
        
        valid_mask = (warped_c_idx[:, 0] >= 0) & (warped_c_idx[:, 0] < Wc) & \
                     (warped_c_idx[:, 1] >= 0) & (warped_c_idx[:, 1] < Hc)
        
        i_valid = torch.arange(L, device=device)[valid_mask]
        j_valid = warped_c_idx[valid_mask, 1] * Wc + warped_c_idx[valid_mask, 0]
        
        conf_matrix_gt[b, i_valid, j_valid] = 1.0
        
        b_ids.append(torch.full_like(i_valid, b))
        i_ids.append(i_valid)
        j_ids.append(j_valid)
        
    batch['conf_matrix_gt'] = conf_matrix_gt
    
    batch['spv_b_ids'] = torch.cat(b_ids) if len(b_ids) > 0 else torch.empty(0, device=device, dtype=torch.long)
    batch['spv_i_ids'] = torch.cat(i_ids) if len(i_ids) > 0 else torch.empty(0, device=device, dtype=torch.long)
    batch['spv_j_ids'] = torch.cat(j_ids) if len(j_ids) > 0 else torch.empty(0, device=device, dtype=torch.long)
    
    num_valid_matches = len(batch['spv_b_ids'])
    batch['conf_matrix_error_gt'] = torch.ones(num_valid_matches, device=device)
    batch['spv_w_pt0_i'] = torch.ones(num_valid_matches, device=device)

    # Dummy tensors to bypass the fine-level errors
    batch['conf_matrix_f'] = torch.zeros(1, 1, 1, device=device)
    batch['conf_matrix_f_gt'] = torch.zeros(1, 1, 1, device=device)
    batch['conf_matrix_f_error_gt'] = torch.zeros(1, 1, 1, device=device)
    batch['expec_f'] = torch.zeros(1, 2, device=device)
    batch['expec_f_gt'] = torch.zeros(1, 2, device=device)

# -------------------------------------------------------------------------
# 4. THE STANDALONE TRAINING LOOP WITH VALIDATION
# -------------------------------------------------------------------------
EPOCHS = 25 
best_val_loss = float('inf')

model.train()
model.coarse_matching.eval() 
model.fine_matching.eval()   

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    total_train_loss = 0.0
    pbar_train = tqdm(train_loader, desc=f"Stage 2 Epoch {epoch+1}/{EPOCHS}")
    
    for batch in pbar_train:
        for k in ['image0', 'image1', 'H_gt']:
            batch[k] = batch[k].cuda()
            
        optimizer.zero_grad()
        
        with autocast('cuda'): 
            model(batch) 
            custom_spvs_homography(batch)
            loss_fn(batch)
            loss = batch['loss'] 
        
        scaler.scale(loss).backward()
        
        # Gradient clipping for safety
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item()
        pbar_train.set_postfix({'Loss': f"{loss.item():.4f}"})
        
        del batch
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --- VALIDATION PHASE ---
    model.eval() 
    total_val_loss = 0.0
    
    with torch.no_grad(): 
        for val_batch in val_loader:
            for k in ['image0', 'image1', 'H_gt']:
                val_batch[k] = val_batch[k].cuda()
                
            with autocast('cuda'):
                model(val_batch)
                custom_spvs_homography(val_batch)
                loss_fn(val_batch)
                val_loss = val_batch['loss']
                
            total_val_loss += val_loss.item()
            del val_batch
            
    avg_val_loss = total_val_loss / len(val_loader)
    
    print(f"End of Stage 2 Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # --- CHECKPOINT SAVING ---
    if avg_val_loss < best_val_loss:
        print(f"--> Stage 2 Validation loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f}. Saving best model!")
        best_val_loss = avg_val_loss
        checkpoint_path = "eloftr_roadscene_stage2_best.pt"
        torch.save(model.state_dict(), checkpoint_path)
    else:
        print(f"--> Stage 2 Validation loss did not improve.")
    
    # Prepare model for the next training epoch
    model.train()
    model.coarse_matching.eval() 
    model.fine_matching.eval()
    torch.cuda.empty_cache()
    
print(f"Stage 2 Complete! Best end-to-end validation loss achieved: {best_val_loss:.4f}")

## 7. Qualitative Evaluation: Visualizing Matches from the Fine-Tuned Model

This cell loads the fine-tuned model (`eloftr_roadscene_stage2_best.pt`) and
defines a `draw_matches` function that:
1. Runs inference on a thermal-optical image pair.
2. Filters the matches by a confidence threshold.
3. Draws the matched keypoints and connecting lines across the two images.

We then apply this function to five sample pairs from the test set, to get a
qualitative sense of how well the fine-tuned model matches thermal and
optical images side by side.


In [ ]:
# -------------------------------------------------------------------------
# 1. LOAD THE TRAINED MODEL
# -------------------------------------------------------------------------
cfg = get_cfg_defaults()
cfg.merge_from_other_cfg(user_cfg)
cfg.LOFTR.MATCH_COARSE.TRAIN_COARSE_PERCENT = 0.0
cfg.LOFTR.MATCH_FINE.ENABLED = False
cfg.LOFTR.COARSE.NPE = [480, 640, 480, 640]

def lower_config(yacs_cfg):
    lower_dict = {}
    for k, v in yacs_cfg.items():
        if hasattr(v, 'items'):
            lower_dict[k.lower()] = lower_config(v)
        else:
            lower_dict[k.lower()] = v
    return lower_dict

global_cfg_lower = lower_config(cfg)

print("Loading trained model...")
model = LoFTR(config=global_cfg_lower['loftr']).cuda()
# Load your final weights here!
model.load_state_dict(torch.load('eloftr_roadscene_stage2_best.pt'))
model.eval() # CRITICAL for inference

# -------------------------------------------------------------------------
# 2. INFERENCE & VISUALIZATION FUNCTION
# -------------------------------------------------------------------------
def draw_matches(ir_path, vis_path, conf_threshold=0.2):
    # Load and preprocess images
    img0_raw = cv2.imread(ir_path, cv2.IMREAD_GRAYSCALE)
    img1_raw = cv2.imread(vis_path, cv2.IMREAD_GRAYSCALE)
    
    img0 = cv2.resize(img0_raw, (640, 480))
    img1 = cv2.resize(img1_raw, (640, 480))
    
    img0_t = torch.from_numpy(img0).float()[None, None].cuda() / 255.
    img1_t = torch.from_numpy(img1).float()[None, None].cuda() / 255.
    
    batch = {'image0': img0_t, 'image1': img1_t}
    
    # Run Inference
    with torch.no_grad():
        model(batch)
        
    # Extract coarse matches and confidences
    mkpts0 = batch['mkpts0_c'].cpu().numpy()
    mkpts1 = batch['mkpts1_c'].cpu().numpy()
    mconf  = batch['mconf'].cpu().numpy()
    
    # Filter by confidence threshold
    valid = mconf > conf_threshold
    mkpts0 = mkpts0[valid]
    mkpts1 = mkpts1[valid]
    mconf  = mconf[valid]
    
    print(f"Found {len(mconf)} confident matches!")
    
    # Visualization setup
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].imshow(img0, cmap='gray')
    axes[0].set_title('Infrared (Image 0)')
    axes[0].axis('off')
    
    axes[1].imshow(img1, cmap='gray')
    axes[1].set_title('Visible (Image 1)')
    axes[1].axis('off')
    
    fig.canvas.draw()
    
    # Draw connecting lines
    transFigure = fig.transFigure.inverted()
    for pt0, pt1 in zip(mkpts0, mkpts1):
        # Convert image coordinates to figure coordinates for drawing lines across subplots
        coord0 = axes[0].transData.transform((pt0[0], pt0[1]))
        coord1 = axes[1].transData.transform((pt1[0], pt1[1]))
        fig_coord0 = transFigure.transform(coord0)
        fig_coord1 = transFigure.transform(coord1)
        
        line = plt.Line2D((fig_coord0[0], fig_coord1[0]), (fig_coord0[1], fig_coord1[1]), 
                          transform=fig.transFigure, color='lime', linewidth=1, alpha=0.6)
        fig.add_artist(line)
        
        # Plot points
        axes[0].plot(pt0[0], pt0[1], 'ro', markersize=3)
        axes[1].plot(pt1[0], pt1[1], 'ro', markersize=3)
        
    plt.show()

# -------------------------------------------------------------------------
# 3. TEST IT OUT (5 PAIRS)
# -------------------------------------------------------------------------
print("Visualizing 5 pairs from the test set...")

# Loop through the test loader
for i, test_batch in enumerate(test_loader):
    if i >= 5:  # Stop after 5 pairs
        break
        
    test_fname = test_batch['scene_id'][0]
    print(f"\n--- Pair {i+1}/5: {test_fname} ---")
    
    ir_test_path  = os.path.join(IR_DIR, test_fname)
    vis_test_path = os.path.join(VIS_DIR, test_fname)
    
    # Draw the matches! You can tweak the conf_threshold if there are too many/few lines
    draw_matches(ir_test_path, vis_test_path, conf_threshold=0.1)

## 8. Quantitative Comparison: ORB vs. Pretrained ELoFTR vs. Fine-Tuned ELoFTR

This is the main evaluation cell. It defines:
- `compute_mma`: a Mean Matching Accuracy metric, which measures the
  fraction of matches whose reprojection error (using the ground-truth
  homography) is below a pixel threshold. This serves as our pose /
  reprojection error metric.
- `evaluate_orb`: runs the classical ORB + RANSAC pipeline over the full
  test set.
- `evaluate_eloftr`: runs either the pretrained or the fine-tuned
  EfficientLoFTR model over the full test set, with the same RANSAC
  verification and MMA computation.

Finally, we run all three evaluations (Classical ORB, Pretrained ELoFTR,
Fine-Tuned ELoFTR) over the entire test set and build a comparison table
reporting: average total matches, average RANSAC inliers, inlier ratio,
MMA @ 3px, and matching speed (FPS). This table is the core quantitative
result for the final presentation.


In [ ]:
# -------------------------------------------------------------------------
# 1. EVALUATION METRICS HELPER
# -------------------------------------------------------------------------
def compute_mma(mkpts0, mkpts1, H_gt, threshold=3.0):
    if len(mkpts0) == 0:
        return 0.0
        
    pts0_h = np.concatenate([mkpts0, np.ones((len(mkpts0), 1))], axis=1)
    pts1_proj_h = (H_gt @ pts0_h.T).T
    pts1_proj = pts1_proj_h[:, :2] / (pts1_proj_h[:, 2:] + 1e-8)
    
    errors = np.linalg.norm(mkpts1 - pts1_proj, axis=1)
    accuracy = (errors <= threshold).mean()
    return accuracy

# -------------------------------------------------------------------------
# 2. CLASSICAL BASELINE (ORB)
# -------------------------------------------------------------------------
def evaluate_orb(test_loader):
    print("Evaluating Classical Baseline (ORB)...")
    orb = cv2.ORB_create(nfeatures=2000)
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    
    # --- ADDED TOTAL MATCHES TRACKING ---
    results = {'time': [], 'total_matches': [], 'inliers': [], 'mma': [], 'inlier_ratio': []}
    
    for batch in tqdm(test_loader, desc="ORB"):
        img0 = (batch['image0'][0, 0].numpy() * 255).astype(np.uint8)
        img1 = (batch['image1'][0, 0].numpy() * 255).astype(np.uint8)
        H_gt = batch['H_gt'][0].numpy()
        
        start_time = time.time()
        kp1, des1 = orb.detectAndCompute(img0, None)
        kp2, des2 = orb.detectAndCompute(img1, None)
        
        mkpts0, mkpts1 = [], []
        total_matches = 0
        inliers = 0
        mma = 0.0
        inlier_ratio = 0.0
        
        if des1 is not None and des2 is not None:
            matches = matcher.match(des1, des2)
            total_matches = len(matches)
            
            if len(matches) >= 4:
                mkpts0 = np.float32([kp1[m.queryIdx].pt for m in matches])
                mkpts1 = np.float32([kp2[m.trainIdx].pt for m in matches])
                
                _, mask = cv2.findHomography(mkpts0, mkpts1, cv2.RANSAC, 3.0)
                if mask is not None:
                    valid = mask.ravel() == 1
                    inliers = valid.sum()
                    mma = compute_mma(mkpts0[valid], mkpts1[valid], H_gt)
                    inlier_ratio = inliers / total_matches
        
        results['time'].append(time.time() - start_time)
        results['total_matches'].append(total_matches)
        results['inliers'].append(inliers)
        results['mma'].append(mma)
        results['inlier_ratio'].append(inlier_ratio)
        
    return results

# -------------------------------------------------------------------------
# 3. ELoFTR EVALUATION FUNCTION
# -------------------------------------------------------------------------
def evaluate_eloftr(test_loader, weights_path, desc_name):
    print(f"Evaluating {desc_name}...")
    
    cfg = get_cfg_defaults()
    cfg.merge_from_other_cfg(user_cfg)
    cfg.LOFTR.MATCH_COARSE.TRAIN_COARSE_PERCENT = 0.0
    cfg.LOFTR.MATCH_FINE.ENABLED = False
    cfg.LOFTR.COARSE.NPE = [480, 640, 480, 640]

    def lower_config(yacs_cfg):
        lower_dict = {}
        for k, v in yacs_cfg.items():
            if hasattr(v, 'items'):
                lower_dict[k.lower()] = lower_config(v)
            else:
                lower_dict[k.lower()] = v
        return lower_dict
        
    model = LoFTR(config=lower_config(cfg)['loftr']).cuda()
    state_dict = torch.load(weights_path, weights_only=False)
    
    if 'state_dict' in state_dict:
        model.load_state_dict(state_dict['state_dict'])
    else:
        model.load_state_dict(state_dict)
        
    model.eval()
    
    # --- ADDED TOTAL MATCHES TRACKING ---
    results = {'time': [], 'total_matches': [], 'inliers': [], 'mma': [], 'inlier_ratio': []}
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=desc_name):
            b_cuda = {k: v.cuda() if torch.is_tensor(v) else v for k, v in batch.items()}
            H_gt = batch['H_gt'][0].numpy()
            
            start_time = time.time()
            model(b_cuda)
            
            mkpts0 = b_cuda['mkpts0_c'].cpu().numpy()
            mkpts1 = b_cuda['mkpts1_c'].cpu().numpy()
            
            total_matches = len(mkpts0)
            inliers = 0
            mma = 0.0
            inlier_ratio = 0.0
            
            if total_matches >= 4:
                _, mask = cv2.findHomography(mkpts0, mkpts1, cv2.RANSAC, 3.0)
                if mask is not None:
                    valid = mask.ravel() == 1
                    inliers = valid.sum()
                    mma = compute_mma(mkpts0[valid], mkpts1[valid], H_gt)
                    inlier_ratio = inliers / total_matches
            
            results['time'].append(time.time() - start_time)
            results['total_matches'].append(total_matches)
            results['inliers'].append(inliers)
            results['mma'].append(mma)
            results['inlier_ratio'].append(inlier_ratio)
            
            del b_cuda
            
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return results

# -------------------------------------------------------------------------
# 4. EXECUTE PIPELINE & GENERATE COMPARISON TABLE
# -------------------------------------------------------------------------
res_orb = evaluate_orb(test_loader)
res_pretrained = evaluate_eloftr(test_loader, "./efficientloftr/weights/eloftr_outdoor.ckpt", "Pre-trained ELoFTR")
res_finetuned = evaluate_eloftr(test_loader, "eloftr_roadscene_stage2_best.pt", "Fine-Tuned ELoFTR")

metrics = []
for name, res in zip(["Classical (ORB)", "Pre-trained ELoFTR", "Fine-Tuned ELoFTR"], 
                     [res_orb, res_pretrained, res_finetuned]):
                     
    avg_total = np.mean(res['total_matches'])
    avg_inliers = np.mean(res['inliers'])
    avg_ratio = np.mean(res['inlier_ratio']) * 100 
    avg_mma = np.mean(res['mma']) * 100 
    avg_fps = 1.0 / np.mean(res['time'])
    
    metrics.append({
        "Method": name,
        "Avg Total Matches": f"{avg_total:.1f}", # Added to table
        "Avg RANSAC Inliers": f"{avg_inliers:.1f}",
        "Inlier Ratio (%)": f"{avg_ratio:.1f}%",
        "MMA @ 3px (%)": f"{avg_mma:.2f}%",
        "Speed (FPS)": f"{avg_fps:.1f}"
    })

df_comparison = pd.DataFrame(metrics)

print("\n" + "="*85)
print("PHASE 5: QUANTITATIVE COMPARISON (ROADSCENE TEST SET)")
print("="*85)
display(df_comparison)